# 🕸️ Building Your First LangGraph Conversation Graph

## Learning Objectives
In this notebook, you will learn:
1. **State Definition** - How to define a `TypedDict` state schema with a reducer (`operator.add`) for accumulating messages
2. **Graph Nodes** - How to write node functions that read from and update shared state
3. **Graph Construction** - How to wire nodes together with `StateGraph`, `START`, and `END`
4. **State-Driven Behavior** - How a sentiment-analysis node can influence a downstream response-generation node
5. **Running a Compiled Graph** - How to invoke a compiled LangGraph app end-to-end

## Prerequisites
- Basic understanding of LangGraph concepts (nodes, edges, state)
- Familiarity with LangChain chat models
- An `OPENAI_API_KEY` configured in a `.env` file at the project root

---
## 📦 Part 1: Environment Setup

We import the LangGraph primitives (`StateGraph`, `START`, `END`), the LangChain chat model factory, and the message types used to talk to the LLM. We also load environment variables from a `.env` file so the OpenAI API key is available.

In [1]:
# ============================================================================
# IMPORTS AND SETUP: LangGraph, LangChain, and Environment Variables
# ============================================================================
# Standard library
import operator

# Third-party
from dotenv import load_dotenv
from typing_extensions import Annotated, TypedDict

# LangChain / LangGraph
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph

# Load API keys (OPENAI_API_KEY, etc.) from the .env file
load_dotenv()

print("✅ Environment loaded and imports ready!")

✅ Environment loaded and imports ready!


---
## 🗂️ Part 2: Define the Conversation State

LangGraph passes a shared **state** object between nodes. Here the state tracks the running message history, the detected sentiment of the latest message, and a running count of AI responses.

### Key Concepts:
- **`TypedDict`**: Defines the shape of the state dictionary
- **`Annotated[list, operator.add]`**: Tells LangGraph to *append* new messages to `messages` instead of overwriting the list on every node update

In [2]:
# ============================================================================
# CONVERSATION STATE: Schema for the Graph
# ============================================================================
class ConversationState(TypedDict):
    """Shared state passed between nodes in the conversation graph."""

    messages: Annotated[list, operator.add]  # Accumulates conversation turns
    sentiment: str  # Sentiment detected for the latest human message
    response_count: int  # Running count of AI responses generated

---
## 🔨 Part 3: Build the Conversation Graph

Now we assemble the graph itself. It has two nodes:
1. **`analyze_sentiment`** - Classifies the sentiment of the incoming message as positive, negative, or neutral
2. **`generate_response`** - Picks a system prompt based on that sentiment and generates the AI's reply

The nodes are connected in a simple linear flow: `START → analyze_sentiment → generate_response → END`.

In [3]:
# ============================================================================
# CREATE_CONVERSATION_GRAPH: Assemble Nodes, Edges, and Compile
# ============================================================================
def create_conversation_graph():
    llm = init_chat_model("gpt-4o-mini", temperature=0.7)

    # Define node function
    def analyze_sentiment(state: ConversationState) -> dict:
        """Analyze the sentiment of the last message."""
        last_message = state["messages"][-1]

        response = llm.invoke(
            [
                SystemMessage(
                    content="Classify sentiment as: positive, negative, or neutral. Reply with just the word."
                ),
                HumanMessage(content=last_message),
            ]
        )

        return {"sentiment": response.content.lower().strip()}

    def generate_response(state: ConversationState) -> dict:
        """Generate appropriate response based on sentiment."""
        sentiment = state["sentiment"]
        last_message = state["messages"][-1]

        system_prompts = {
            "positive": "Respond enthusiastically and build on their positive energy.",
            "negative": "Respond empathetically and offer support.",
            "neutral": "Respond helpfully and informatively.",
        }

        prompt = system_prompts.get(sentiment, system_prompts["neutral"])

        response = llm.invoke(
            [SystemMessage(content=prompt), HumanMessage(content=last_message)]
        )

        return {"messages": [f"AI: {response.content}"], "response_count": 1}

    # Create graph
    graph = StateGraph(ConversationState)

    # Add nodes
    graph.add_node("analyze_sentiment", analyze_sentiment)
    graph.add_node("generate_response", generate_response)

    # Add edges
    graph.add_edge(START, "analyze_sentiment")
    graph.add_edge("analyze_sentiment", "generate_response")
    graph.add_edge("generate_response", END)

    app = graph.compile()

    return app

---
## ▶️ Part 4: Run the Demo Conversation

`demo_conversation` compiles the graph and runs it against three sample messages that each land in a different sentiment bucket, printing the detected sentiment and generated response for each turn.

In [4]:
# ============================================================================
# DEMO_CONVERSATION: Drive the Graph with Sample Messages
# ============================================================================
def demo_conversation():
    app = create_conversation_graph()

    # Simulate a conversation
    test_messages = [
        "I just got promoted at work! I'm so excited!",
        "My computer crashed and I lost all my work...",
        "What's the weather like today?",
    ]

    print("Conversation Graph Demo:\n")

    for msg in test_messages:
        result = app.invoke({
            "messages": [f"Human: {msg}"],
            "sentiment": "",
            "response_count": 0
        })

        print(f"Input: {msg}")
        print(f"Sentiment: {result['sentiment']}")
        print(f"Response: {result['messages'][-1]}")
        print("-" * 40)

### 🏁 Running the Demo

The original `__main__` guard is kept as-is below. Jupyter sets `__name__` to `"__main__"`, so this cell runs `demo_conversation()` directly when executed.

In [5]:
# ============================================================================
# RUN: Execute the Demo Conversation
# ============================================================================
if __name__ == "__main__":
    demo_conversation()

Conversation Graph Demo:

Input: I just got promoted at work! I'm so excited!
Sentiment: positive
Response: AI: That’s amazing! Congratulations! 🎉 Your hard work has really paid off, and you totally deserve this recognition! What new responsibilities will you be taking on? I can only imagine how this is going to elevate your career!
----------------------------------------
Input: My computer crashed and I lost all my work...
Sentiment: negative
Response: AI: I'm really sorry to hear that your computer crashed and you lost all your work. That must be incredibly frustrating and disheartening. It’s completely understandable to feel overwhelmed in such a situation. If you’d like, I can help you think through some steps to recover what you can or discuss how to prevent this in the future. You're not alone in this, and I’m here to support you.
----------------------------------------
Input: What's the weather like today?
Sentiment: neutral
Response: AI: I'm unable to provide real-time weathe

---
## 📝 Summary

In this notebook, we learned:

### 1. State and Graph Construction
- **`ConversationState`**: A `TypedDict` state schema using `Annotated[list, operator.add]` to accumulate messages across nodes
- **`StateGraph`**: Wires node functions together with `add_node` and `add_edge`, using `START`/`END` to mark the entry and exit points

### 2. Nodes That Read and Write Shared State
- **`analyze_sentiment`**: Reads the latest message and classifies its sentiment via an LLM call
- **`generate_response`**: Reads the sentiment produced by the previous node and picks a matching system prompt to generate a reply
- Each node returns only the state keys it updates - LangGraph merges these into the overall state

### 3. Running a Compiled Graph
- **`graph.compile()`** produces a runnable `app`
- **`app.invoke(...)`** runs the full node sequence for a given input and returns the final state

### Next Steps
- Move on to the next notebook to explore more advanced LangGraph patterns such as conditional routing and tool-calling nodes
- Experiment with adding new sentiment categories or additional nodes to this graph